# google_cloud_bigquery smoke test

Smoke test for `google-cloud-bigquery`, the Python client for Google BigQuery.
Every example runs offline: it builds schemas, job configurations, table
definitions and query parameters, and constructs a client with anonymous
credentials. No request is sent to BigQuery.

## Example 1: Schema fields and their REST representation

In [ ]:
from google.cloud import bigquery

# A nested, repeated RECORD field, converted to the REST API form and back.
x = bigquery.SchemaField("x", "INTEGER", mode="REQUIRED", description="a counter")
assert x.to_api_repr() == {"name": "x", "type": "INTEGER", "mode": "REQUIRED", "description": "a counter"}

rec = bigquery.SchemaField("rec", "RECORD", mode="REPEATED",
                           fields=[bigquery.SchemaField("label", "STRING"), x])
api = rec.to_api_repr()
print("REST form:", api)
assert api["type"] == "RECORD" and api["mode"] == "REPEATED"
assert [f["name"] for f in api["fields"]] == ["label", "x"]
assert api["fields"][0]["mode"] == "NULLABLE"  # the default mode

back = bigquery.SchemaField.from_api_repr(api)
assert back == rec
assert [f.name for f in back.fields] == ["label", "x"]
print("SchemaField round trip OK")

## Example 2: Query job configuration

In [ ]:
from google.cloud import bigquery

# A dry-run job config with a named query parameter; to_api_repr() is the body sent with a job.
cfg = bigquery.QueryJobConfig(
    dry_run=True,
    use_query_cache=False,
    labels={"team": "demo"},
    query_parameters=[bigquery.ScalarQueryParameter("min_n", "INT64", 5)],
)
assert cfg.dry_run is True and cfg.use_query_cache is False
api = cfg.to_api_repr()
print("job configuration:", api)
assert api["dryRun"] is True
assert api["labels"] == {"team": "demo"}
assert api["query"]["useQueryCache"] is False
assert api["query"]["parameterMode"] == "NAMED"
assert api["query"]["queryParameters"][0] == {
    "name": "min_n", "parameterType": {"type": "INT64"}, "parameterValue": {"value": "5"},
}
print("QueryJobConfig OK")

## Example 3: Table and dataset references

In [ ]:
from google.cloud import bigquery

ref = bigquery.TableReference.from_string("demo.ds.tbl")
print("parsed:", ref.project, ref.dataset_id, ref.table_id)
assert (ref.project, ref.dataset_id, ref.table_id) == ("demo", "ds", "tbl")
assert ref.path == "/projects/demo/datasets/ds/tables/tbl"

# A two-part id takes the project from default_project.
short = bigquery.TableReference.from_string("ds.tbl", default_project="other")
assert str(short) == "other.ds.tbl"

# The same table reached through a DatasetReference compares equal.
assert bigquery.DatasetReference("demo", "ds").table("tbl") == ref

try:
    bigquery.TableReference.from_string("not-a-table-id")
    raise AssertionError("expected ValueError for a bare table id with no default project")
except ValueError as e:
    print("invalid id ->", e)
print("references OK")

## Example 4: Table definition with partitioning and clustering

In [ ]:
from google.cloud import bigquery

table = bigquery.Table(
    "demo.ds.events",
    schema=[bigquery.SchemaField("ts", "TIMESTAMP"), bigquery.SchemaField("n", "INT64")],
)
table.time_partitioning = bigquery.TimePartitioning(type_=bigquery.TimePartitioningType.DAY, field="ts")
table.clustering_fields = ["n"]

api = table.to_api_repr()
print("table resource:", api)
assert api["tableReference"] == {"projectId": "demo", "datasetId": "ds", "tableId": "events"}
assert api["timePartitioning"] == {"type": "DAY", "field": "ts"}
assert api["clustering"] == {"fields": ["n"]}

back = bigquery.Table.from_api_repr(api)
assert back.time_partitioning.field == "ts"
assert back.clustering_fields == ["n"]
assert [f.name for f in back.schema] == ["ts", "n"]
print("Table round trip OK")

## Example 5: Client, query parameters and result rows

In [ ]:
from google.auth.credentials import AnonymousCredentials
from google.cloud import bigquery

# Constructing a client does not contact BigQuery; requests only start when a method is called.
client = bigquery.Client(project="demo", credentials=AnonymousCredentials())
print("client project:", client.project)
assert client.project == "demo"

ids = bigquery.ArrayQueryParameter("ids", "INT64", [1, 2, 3]).to_api_repr()
assert ids["parameterType"] == {"type": "ARRAY", "arrayType": {"type": "INT64"}}
assert [v["value"] for v in ids["parameterValue"]["arrayValues"]] == ["1", "2", "3"]
scalar = bigquery.ScalarQueryParameter("min_n", "INT64", 5)
assert bigquery.ScalarQueryParameter.from_api_repr(scalar.to_api_repr()).value == 5

# Row is the type query results come back as: access by name, by attribute, or by position.
row = bigquery.Row(("widget", 7), {"name": 0, "n": 1})
assert row["name"] == "widget" and row.n == 7 and row[1] == 7
assert row.get("missing", "default") == "default"
assert dict(row.items()) == {"name": "widget", "n": 7}
client.close()
print("client, parameters and rows OK")